# Options Pricing Engine — Full Analysis

End-to-end walkthrough: Black-Scholes closed-form pricing and Greeks, Monte Carlo simulation with variance reduction, Longstaff-Schwartz American option pricing, implied volatility recovery, and validation against real AAPL market data.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from core.black_scholes import BlackScholes, check_put_call_parity
from core.monte_carlo import MonteCarloPricer
from core.longstaff_schwartz import LongstaffSchwartz
from core.implied_vol import ImpliedVolCalculator
from validation.convergence import run_convergence_analysis
from visualisation import plots

S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20

## 1. Black-Scholes closed-form pricing and Greeks

In [2]:
bs_call = BlackScholes(S, K, T, r, sigma, "call")
bs_put = BlackScholes(S, K, T, r, sigma, "put")

print(f"Call price: {bs_call.price():.4f}")
print(f"Put price:  {bs_put.price():.4f}")
print(f"Call Greeks: {bs_call.all_greeks()}")

assert check_put_call_parity(S, K, T, r, sigma)
print("Put-call parity holds within 1e-6.")

Call price: 10.4506
Put price:  5.5735
Call Greeks: {'delta': np.float64(0.6368306511756191), 'gamma': np.float64(0.018762017345846895), 'vega': np.float64(0.3752403469169379), 'theta': np.float64(-0.01757267820941972), 'rho': np.float64(0.5323248154537634)}
Put-call parity holds within 1e-6.


## 2. Monte Carlo pricing with variance reduction

In [3]:
mc = MonteCarloPricer(S, K, T, r, sigma, option_type="call", seed=42)

mc_cv = mc.price_european(n_simulations=100_000, use_control_variate=True)
mc_naive = mc.price_european(n_simulations=100_000, use_control_variate=False)

print(f"MC (antithetic + control variate): {mc_cv.price:.4f} (SE={mc_cv.std_error:.5f})")
print(f"MC (antithetic only):               {mc_naive.price:.4f} (SE={mc_naive.std_error:.5f})")
print(f"Variance reduction: {(1 - (mc_cv.std_error / mc_naive.std_error) ** 2) * 100:.1f}% lower SE^2")

asian_fixed = mc.price_asian(n_simulations=100_000, n_steps=252, strike_type="fixed")
print(f"Asian fixed-strike call: {asian_fixed.price:.4f}")

MC (antithetic + control variate): 10.4602 (SE=0.01783)
MC (antithetic only):               10.4853 (SE=0.04687)
Variance reduction: 85.5% lower SE^2


Asian fixed-strike call: 5.7802


## 3. Monte Carlo convergence analysis

In [4]:
convergence_df = run_convergence_analysis(S, K, T, r, sigma, "call")
convergence_df

,n_simulations,price_naive,std_error_naive,abs_error_naive,price_cv,std_error_cv,abs_error_cv
0,1000,10.000398,0.444738,0.450186,10.482972,0.184982,0.032388
1,5000,10.506017,0.209759,0.055433,10.427947,0.080790,0.022637
2,10000,10.539351,0.148789,0.088767,10.423276,0.056029,0.027307
3,50000,10.497039,0.066135,0.046456,10.434746,0.025235,0.015838
4,100000,10.480392,0.046782,0.029808,10.479940,0.017793,0.029356


## 4. Longstaff-Schwartz American put pricing

In [5]:
lsm = LongstaffSchwartz(S, K, T, r, sigma, n_paths=50_000, n_steps=50, basis="laguerre", degree=3, seed=42)
american_price, boundary_df = lsm.price()
european_price = bs_put.price()

print(f"European put: {european_price:.4f}")
print(f"American put: {american_price:.4f}")
print(f"Early exercise premium: {american_price - european_price:.4f}")
boundary_df.head()

European put: 5.5735
American put: 6.0510
Early exercise premium: 0.4774


,time,critical_price
0,0.08,84.660061
1,0.12,84.583440
2,0.14,84.449693
3,0.16,84.556849
4,0.18,84.401083


## 5. Implied volatility recovery

In [6]:
iv_calc = ImpliedVolCalculator()
market_price = bs_call.price()
recovered_sigma = iv_calc.solve(market_price, S, K, T, r, "call")
print(f"True sigma: {sigma}, recovered: {recovered_sigma:.6f}")

True sigma: 0.2, recovered: 0.200000


## 6. Visualisations

Generates all plots into `../results/` (Greeks surfaces, vol smile, early exercise boundary, American vs European premium).

In [7]:
plots.plot_greeks_surfaces(n_points=20)
plots.plot_vol_smile()
plots.plot_early_exercise_boundary(n_paths=20_000)
plots.plot_american_vs_european_premium(n_paths=10_000, n_points=10)
print("All plots saved to results/")

All plots saved to results/


## 7. Real market validation (AAPL)

Requires live network access. See `validation/market_validation.py` for the standalone script and `results/market_validation.csv` / `results/summary.txt` for cached output.

In [8]:
from validation.market_validation import build_validation_table, build_vol_smile

try:
    comparison_df, summary_stats = build_validation_table("AAPL", n_expiries=3)
    print(summary_stats)
    comparison_df.head()
except (ConnectionError, TimeoutError, OSError) as e:
    print(f"Network unavailable: {e}")

{'n_options': 185, 'spot': 297.010009765625, 'risk_free_rate': 0.03657999992370606, 'realized_vol': 0.21970695503876933, 'mae': 1.2608762053745013, 'rmse': 2.8847666193846444, 'mean_pct_error': -53.261644430268916, 'mean_implied_vol': 0.997831038656453, 'min_implied_vol': 0.19311500264696232, 'max_implied_vol': 4.184962234497071}
